In [ ]:
# Colab/bootstrap: clone this repository and install it editable.
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/JeonDongJun/mindscopex_analysis"
MARK_REL = Path("src") / "mindscopex_analysis" / "__init__.py"


def find_repo_root(start=None):
    candidate = Path(start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / MARK_REL).is_file():
            return path
    return None


root = find_repo_root()
if root is None:
    workdir = Path(os.environ.get("COLAB_REPO_DIR", "/content/mindscopex_analysis"))
    if (workdir / MARK_REL).is_file():
        subprocess.run(["git", "-C", str(workdir), "pull", "--ff-only"], check=False)
        root = workdir
    else:
        workdir.parent.mkdir(parents=True, exist_ok=True)
        if workdir.exists():
            shutil.rmtree(workdir)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(workdir)])
        root = workdir

os.environ["MINDSCOPEX_ROOT"] = str(root.resolve())
os.chdir(root)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
print("ready:", root)


# 00. Qwen CRT 실제 텍스트 답변

Qwen 모델군에 CRT 문제를 직접 제시하고, thinking/non-thinking 모드에서 생성되는 전체 텍스트를 확인하는 기준 실험입니다. 두 모드 모두 보이는 최종 출력은 짧은 답만 요구하며, thinking 모드에서는 reasoning이 생성된 `<think>...</think>` 안에만 있는지 함께 검사합니다.

## 실행 전 고려사항

1. 기본 모델은 hybrid post-trained `Qwen3-1.7B`, `4B`, `8B`이며 한 번에 하나만 GPU에 올립니다. `0.6B`는 형식 준수 stress test로만 선택 실행합니다.
2. 시스템 프롬프트는 reasoning 자체를 제한하지 않고, reasoning 뒤의 최종 응답만 짧은 답과 단위로 고정합니다.
3. 샘플링은 [Qwen3 모델 카드](https://huggingface.co/Qwen/Qwen3-1.7B)의 권장값을 사용하고 seed를 기록합니다. 샘플링 결과는 seed에 따라 달라질 수 있습니다.
4. `think_protocol_ok=False`이면 `protocol_issue`가 누락 태그, 빈 reasoning, 잘림, 최종 답 누락 중 원인을 표시합니다.
5. `answer_only`와 `format_issue`는 thinking protocol과 별개로 최종 응답이 답만 포함하는지 검사합니다.
6. `answer_label`은 최종 답변 텍스트의 단순 문자열 판정입니다. `both`와 `other`는 반드시 원문을 직접 확인합니다.
7. 주 mechanistic 대상은 `Qwen3-8B`와 공식 `Qwen3-8B-Base` SAE의 조합을 권장합니다. Base SAE를 post-trained checkpoint에 옮길 때는 reconstruction을 먼저 검증해야 합니다.
8. `Qwen3.5-27B`는 모델 자체에 맞춘 공식 SAE가 있지만 현재 실행 환경과 L40 비용을 고려해 후속 단계로 둡니다.
9. `truncated=True`이면 reasoning이 끝나지 않은 것이므로 `MAX_NEW_TOKENS`를 늘려 다시 실행합니다.


In [ ]:
import os
import sys
from pathlib import Path

root = Path(os.environ.get("MINDSCOPEX_ROOT", Path.cwd())).resolve()
if not (root / "src" / "mindscopex_analysis" / "__init__.py").is_file():
    for candidate in [root, *root.parents]:
        if (candidate / "src" / "mindscopex_analysis" / "__init__.py").is_file():
            root = candidate
            break
    else:
        raise RuntimeError("Could not find repository root. Run the clone cell first.")

src_path = str(root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(root)


In [ ]:
from collections import Counter
from html import escape

from IPython.display import HTML, display

from mindscopex_analysis import (
    CRT_FINAL_ANSWER_SYSTEM_PROMPT,
    DEFAULT_QWEN_CHAT_MODEL_IDS,
    QWEN_FORMAT_STRESS_MODEL_IDS,
    RECOMMENDED_INTERPRETABILITY_MODEL_ID,
    RECOMMENDED_INTERPRETABILITY_SAE_REPO_ID,
    clear_device_cache,
    crt_transfer_cases,
    generate_crt_response_suite,
    load_qwen_text_generation_model,
    recommended_dtype_name,
    save_qwen_text_responses,
)


def display_records(rows, columns):
    head = "".join(f"<th>{escape(str(column))}</th>" for column in columns)
    body = []
    for row in rows:
        cells = "".join(f"<td>{escape(str(row.get(column, '')))}</td>" for column in columns)
        body.append(f"<tr>{cells}</tr>")
    table = (
        "<div style='overflow-x:auto'><table style='border-collapse:collapse'>"
        f"<thead><tr>{head}</tr></thead><tbody>{''.join(body)}</tbody></table></div>"
    )
    display(HTML(table))


In [ ]:
INCLUDE_0_6B_FORMAT_STRESS = False
model_ids = list(DEFAULT_QWEN_CHAT_MODEL_IDS)
if INCLUDE_0_6B_FORMAT_STRESS:
    model_ids = [*QWEN_FORMAT_STRESS_MODEL_IDS, *model_ids]

MODEL_SPECS = [
    {
        "model_id": model_id,
        "thinking_modes": (False, True),
        "use_chat_template": True,
    }
    for model_id in model_ids
]

INCLUDE_QWEN_SCOPE_BASE = False
if INCLUDE_QWEN_SCOPE_BASE:
    MODEL_SPECS.append(
        {
            "model_id": "Qwen/Qwen3-8B-Base",
            "thinking_modes": (None,),
            "use_chat_template": False,
        }
    )

CASES = crt_transfer_cases()
DTYPE = recommended_dtype_name()
MAX_NEW_TOKENS = 4096
DO_SAMPLE = True
SEED = 42
SYSTEM_PROMPT = CRT_FINAL_ANSWER_SYSTEM_PROMPT
OUTPUT_PATH = root / "outputs" / "00_qwen_crt_text_responses.json"

print({
    "models": [spec["model_id"] for spec in MODEL_SPECS],
    "cases": [case.case_id for case in CASES],
    "dtype": DTYPE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": DO_SAMPLE,
    "seed": SEED,
    "system_prompt": SYSTEM_PROMPT,
    "recommended_analysis_model": RECOMMENDED_INTERPRETABILITY_MODEL_ID,
    "recommended_sae": RECOMMENDED_INTERPRETABILITY_SAE_REPO_ID,
})


In [ ]:
case_rows = [
    {
        "case": case.case_id,
        "family": case.family,
        "correct": case.correct_answer.strip(),
        "lure": case.lure_answer.strip(),
        "prompt": case.prompt,
    }
    for case in CASES
]
display_records(case_rows, ["case", "family", "correct", "lure", "prompt"])


In [ ]:
responses = []

for spec in MODEL_SPECS:
    model_id = spec["model_id"]
    print(f"\nLoading {model_id} ...")
    model, tokenizer = load_qwen_text_generation_model(
        model_id,
        device_map="auto",
        dtype=DTYPE,
    )

    model_responses = generate_crt_response_suite(
        model,
        tokenizer,
        CASES,
        model_id=model_id,
        thinking_modes=spec["thinking_modes"],
        use_chat_template=spec["use_chat_template"],
        system_prompt=SYSTEM_PROMPT,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=DO_SAMPLE,
        seed=SEED,
    )
    responses.extend(model_responses)
    save_qwen_text_responses(responses, OUTPUT_PATH)

    for response in model_responses:
        preview = response.answer.replace("\n", " ")[:120]
        print(f"[{response.case_id} | {response.mode} | {response.answer_label}] {preview}")

    del model_responses, model, tokenizer
    clear_device_cache()

print(f"\nsaved {len(responses)} responses to {OUTPUT_PATH}")


In [ ]:
summary_rows = []
for response in responses:
    row = response.summary_row()
    row["final_answer"] = row["final_answer"].replace("\n", " ")[:200]
    summary_rows.append(row)

display_records(
    summary_rows,
    [
        "model", "case", "mode", "think_block", "reasoning_chars",
        "think_protocol_ok", "protocol_issue", "answer_only",
        "format_issue", "label", "final_answer", "output_tokens",
        "seconds", "truncated",
    ],
)


In [ ]:
SELECT_MODEL = ""  # 예: Qwen3-1.7B, 빈 문자열이면 전체
SELECT_CASE = ""   # 예: bat_ball_original, 빈 문자열이면 전체
SHOW_THINKING = True

selected = [
    response
    for response in responses
    if (not SELECT_MODEL or response.model_id.endswith(SELECT_MODEL))
    and (not SELECT_CASE or response.case_id == SELECT_CASE)
]

for response in selected:
    heading = (
        f"{response.model_id} | {response.case_id} | "
        f"{response.mode} | {response.answer_label}"
    )
    sections = [f"<h3>{escape(heading)}</h3>"]
    protocol = (
        f"thinking block: {response.has_thinking_block}, "
        f"reasoning chars: {len(response.thinking)}, "
        f"protocol ok: {response.thinking_protocol_ok}, "
        f"protocol issue: {response.thinking_protocol_issue}, "
        f"answer only: {response.final_answer_format_ok}, "
        f"format issue: {response.final_answer_format_issue}"
    )
    sections.append(f"<p>{escape(protocol)}</p>")
    if SHOW_THINKING and response.thinking:
        sections.append(f"<h4>Thinking</h4><pre>{escape(response.thinking)}</pre>")
    sections.append(f"<h4>Final answer</h4><pre>{escape(response.answer)}</pre>")
    display(HTML("".join(sections)))


In [ ]:
counts = Counter(
    (response.model_id.rsplit("/", 1)[-1], response.mode, response.answer_label)
    for response in responses
)
aggregate_rows = [
    {"model": model, "mode": mode, "label": label, "count": count}
    for (model, mode, label), count in sorted(counts.items())
]
display_records(aggregate_rows, ["model", "mode", "label", "count"])

protocol_rows = [
    {
        "model": response.model_id.rsplit("/", 1)[-1],
        "case": response.case_id,
        "mode": response.mode,
        "think_block": response.has_thinking_block,
        "reasoning_chars": len(response.thinking),
        "protocol_ok": response.thinking_protocol_ok,
        "protocol_issue": response.thinking_protocol_issue,
        "answer_only": response.final_answer_format_ok,
        "format_issue": response.final_answer_format_issue,
        "truncated": response.hit_max_tokens,
        "final_words": len(response.answer.split()),
        "final_answer": response.answer.replace("\n", " ")[:120],
        "raw_preview": response.raw_text.replace("\n", " ")[:240],
    }
    for response in responses
]
display_records(
    protocol_rows,
    [
        "model", "case", "mode", "think_block", "reasoning_chars",
        "protocol_ok", "protocol_issue", "answer_only",
        "format_issue", "truncated", "final_words", "final_answer",
    ],
)

protocol_failures = [row for row in protocol_rows if row["protocol_ok"] is False]
format_failures = [row for row in protocol_rows if not row["answer_only"]]
print("thinking protocol failures:", len(protocol_failures))
if protocol_failures:
    display_records(
        protocol_failures,
        ["model", "case", "mode", "protocol_issue", "truncated", "raw_preview"],
    )
print("final-answer format failures:", len(format_failures))
if format_failures:
    display_records(
        format_failures,
        ["model", "case", "mode", "format_issue", "final_answer"],
    )


## 결과를 읽는 순서

1. 먼저 `truncated`가 없는지 확인합니다. 잘린 응답은 정오 판정에서 제외합니다.
2. thinking 응답은 `think_protocol_ok=True`여야 하며, 실패했다면 `protocol_issue`와 `raw_preview`로 원인을 확인합니다.
3. non-thinking 응답은 thinking tag가 없어야 합니다. `answer_only=False`는 별개의 최종 출력 형식 실패입니다.
4. `truncated_before_think_close`는 출력 길이 문제이고, `missing_thinking_block`은 모델의 protocol 불이행이므로 같은 실패로 합치지 않습니다.
5. `both`는 정답과 함정 답을 최종 출력에서 함께 언급한 경우이므로 시스템 프롬프트를 따르지 않은 응답으로 직접 확인합니다.
6. 같은 모델의 thinking/non-thinking 차이를 비교해 reasoning이 함정 답을 수정하는지 확인합니다.
7. `0.6B`는 주 결과에 합치지 않고 형식 준수 stress test 또는 부록으로만 보고합니다.
8. 이후 feature 실험은 `Qwen3-8B`에서 함정 답이 실제로 관찰되거나 thinking 여부에 따라 답이 바뀌는 문항을 우선 대상으로 삼습니다.
9. 확률적 결과를 논문에 사용할 때는 `SEED`를 여러 개로 늘려 정답률과 함정 답률의 평균 및 신뢰구간을 계산합니다.
